# Appendix 10.9: Model Context Protocol (MCP) Basics

- [Lesson](#lesson)
- [Exercise](#exercise)

## Setup

This appendix is conceptual and needs no API key — MCP is about *how tools get supplied to Claude*, not a new prompting technique, so there's nothing new to call `client.messages.create()` with. The exercise is a pure Python data-transformation exercise.

---

## Lesson

### The problem MCP solves

In Appendix 10.2 you hand-wrote a `tools` list — one JSON Schema per function, by hand, specific to your app. That's fine for one app with a handful of tools. Now imagine the real world: dozens of AI apps (Claude Desktop, Claude Code, IDEs, other agent frameworks) each want access to dozens of external systems (GitHub, Slack, databases, internal APIs). Without a standard, that's every app hand-writing a custom integration for every system — an **M × N** problem that scales terribly.

**MCP (Model Context Protocol)**, an open protocol introduced by Anthropic in late 2024 and since adopted broadly across the industry, turns that into an **M + N** problem: each external system implements *one* MCP server (once), and each AI app implements *one* MCP client (once). Any MCP client can then talk to any MCP server, the same way any web browser can talk to any website because both sides speak HTTP.

**You've been using MCP this whole session.** Every tool in this conversation prefixed `mcp__` — the browser tools, the calendar tools, the file-search tools — is being supplied to Claude Code through an MCP server, discovered and wired up automatically instead of hand-coded. If you understand Appendix 10.2's tool-use loop, you already understand what MCP is doing under the hood: it's discovering `tools` list entries for you and routing the resulting `tool_use` calls to the right server, instead of you writing that schema and routing logic by hand.

**Where this shows up in real systems:**
- Claude Desktop or Claude Code connecting to a local filesystem, a GitHub account, or a company's internal Slack/Jira/database via MCP servers, configured once and reused across every conversation.
- Third-party companies (as you can see from the plugin/connector list available in this very session — Slack, Notion, Linear, Asana, and more) shipping an MCP server as *the* integration surface for their product, instead of building bespoke integrations per AI vendor.
- Internal platform teams standardizing how every AI feature in their company accesses internal systems, instead of re-solving auth, schema definition, and routing for every new agent project.

### The three things an MCP server can expose
- **Tools** — callable functions, exactly like the `tools` list from Appendix 10.2.
- **Resources** — readable data (files, records) the host app can pull into context.
- **Prompts** — reusable, parameterized prompt templates the server author has pre-written.

Tools are what you'll interact with most, and they map directly onto everything you learned in Appendix 10.2 — an MCP tool becomes a `tools` entry, a call to it becomes a `tool_use` block, and its result comes back as a `tool_result`, same as any hand-written tool.

### What an MCP tool definition looks like

MCP tools are discovered dynamically: a client asks a server "what tools do you have?" (`tools/list`) and gets back a JSON description very close to — but not byte-identical to — what `messages.create` expects. The main practical difference is naming convention: MCP (a JSON-RPC-based protocol) uses `inputSchema` (camelCase), while the Anthropic Messages API uses `input_schema` (snake_case). A host app's MCP client handles this translation for you; below, we do it by hand once so you can see there's no magic involved.

In [ ]:
# What an MCP server's tools/list response looks like (simplified)
mcp_list_tools_response = {
    "tools": [
        {
            "name": "search_issues",
            "description": "Search project issues by keyword and status.",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Keyword to search for."},
                    "status": {"type": "string", "enum": ["open", "closed", "any"]}
                },
                "required": ["query"]
            }
        }
    ]
}
print(mcp_list_tools_response)

---

## Exercise

### Exercise 10.9.1 - MCP-to-Anthropic Tool Adapter

Write `mcp_tools_to_anthropic_tools(mcp_response)` that converts an MCP `tools/list` response (like `mcp_list_tools_response` above) into the `tools` list format from Appendix 10.2 — a list of dicts with `name`, `description`, and `input_schema` (snake_case) keys, ready to pass straight into `client.messages.create(tools=...)`.

In [ ]:
# Your function goes here — this is the only field you should change
def mcp_tools_to_anthropic_tools(mcp_response):
    pass

result = mcp_tools_to_anthropic_tools(mcp_list_tools_response)
print(result)

print("\n--------------------------- GRADING ---------------------------")
checks = [
    isinstance(result, list) and len(result) == 1,
    result[0].get("name") == "search_issues",
    "input_schema" in result[0],
    "inputSchema" not in result[0],
    result[0].get("input_schema", {}).get("required") == ["query"],
]
print("All checks passed!" if all(checks) else f"Checks: {checks}")

❓ If you want a hint, run the cell below!

In [ ]:
from hints import exercise_10_9_1_hint; print(exercise_10_9_1_hint)

### Congrats!

**Recap:** MCP is a protocol, not a new prompting technique — it standardizes *how* tools, data, and prompt templates get discovered and supplied to an AI app, so integrations get built once per system instead of once per (app, system) pair. Once an MCP tool reaches Claude, it's handled by exactly the tool-use mechanics from Appendix 10.2: a `tools` entry, a `tool_use` request, a `tool_result` response.

**Interview-answer framing:** "MCP is an open protocol, introduced by Anthropic, that standardizes how AI applications connect to external tools and data sources — it turns what used to be an M-times-N custom integration problem (every app hand-building every integration) into an M-plus-N problem, where each system ships one MCP server and each app ships one MCP client. It doesn't change the underlying tool-use mechanics at all — an MCP tool still shows up to the model as a standard tool-use call — it just standardizes discovery and routing so you're not hand-writing bespoke integrations for every external system you want an agent to reach."

That's the end of the appendix — and the course. You've gone from writing a single well-structured prompt to understanding the full stack a production Claude application is built on: prompting fundamentals, tool use, caching, reasoning, structured output, grounding, agentic loops, and the protocol that wires it all together.